# Uno Coefficient Relationships, Short Version

This notebook keeps the current Colab GPU workflow while importing the reusable Uno formulas and plotting helpers from Python files. The original long notebook is left intact.

Reusable code locations:

- `src/aberration_simulation/uno_conventions.py`: value definitions, phase conventions, under/over pairing helpers.
- `scripts/plots_uno_convention.py`: relationship, coupled-response, and summary plots.
- `scripts/plot_probe_shapes.py`: probe-shape galleries.

The pairwise uniqueness diagnostics remain inline here for active inspection and adjustment.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")

## 4. Imports and Current Conventions

In [ ]:
from pathlib import Path
import csv
import sys
import zipfile

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(ROOT / "scripts"))

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp
from aberration_simulation.line_profiles import extract_line_profiles_from_stack
from aberration_simulation.optics import SimulationConfig, run_simulation, save_npz
from aberration_simulation.uno_conventions import (
    PRIMARY_PHASE_CONVENTIONS,
    UNO_HARMONIC_ORDERS,
    add_complex_columns,
    compute_line_characteristics,
    compute_uno_values,
    select_under_over_pairs,
)
from plots_uno_convention import (
    plot_a1_s3_coupling_maps,
    plot_c1_c3_coupling_maps,
    plot_relationship_summary,
    plot_relationships,
)
from plot_probe_shapes import plot_probe_shape_galleries

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())
print("Uno primary phase conventions:")
for name, convention in PRIMARY_PHASE_CONVENTIONS.items():
    period = 360.0 / UNO_HARMONIC_ORDERS[name]
    print(f"  {name}: sign={convention['sign']:g}, offset={convention['offset_deg']:g} deg, period={period:g} deg")

## 5. Build One-Coefficient-at-a-Time Sweep

In [ ]:
OUTPUT_DIR = ROOT / "outputs" / "uno_relationships"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
C1_OFFSETS = [UNDER_FOCUS_C1_OFFSET, OVER_FOCUS_C1_OFFSET]
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

C1_C3_GRID_LABEL = "C1_C3_grid"
C1_C3_GRID_C1_VALUES = [-80, -40, -20, 0, 20, 40, 80]
C1_C3_GRID_C3_VALUES = [0, 0.1, 0.2, 0.5, 1.0, 1.5, 2.0]

A1_S3_GRID_LABEL = "A1_S3_grid"
A1_S3_GRID_A1_AMP_VALUES = [0, 30, 60]
A1_S3_GRID_S3_AMP_VALUES = [0, 8, 16]
A1_S3_GRID_A1_PHASE_VALUES = [0, 45, 90, 135]
A1_S3_GRID_S3_PHASE_VALUES = [0, 45, 90, 135]

BASELINE_PARAMETERS = {
    "C1_offset": 0,
    "A3_amp": 0,
    "S3_amp": 0,
    "A2_amp": 0,
    "B2_amp": 0,
    "C1": 0,
    "C3": 0,
    "A1_amp": 0,
    "A1_phase": 0,
    "A2_phase": 0,
    "A3_phase": 0,
    "S3_phase": 0,
    "B2_phase": 0,
}

SCALAR_SWEEP_SPECS = [
    {
        "label": "C1",
        "value_name": "C1_value",
        "input_field": "C1",
        "input_values": [-80, -40, -20, 20, 40, 80],
    },
    {
        "label": "C3",
        "value_name": "C3_value",
        "input_field": "C3",
        "input_values": [0.1, 0.2, 0.5, 1.0, 1.5, 2.0],
    },
]

HARMONIC_SWEEP_SPECS = [
    {
        "label": "A1",
        "value_name": "A1_value",
        "amp_field": "A1_amp",
        "phase_field": "A1_phase",
        "amp_values": [5, 15, 30, 60],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
    {
        "label": "B2/C21",
        "value_name": "B2_value",
        "amp_field": "B2_amp",
        "phase_field": "B2_phase",
        "amp_values": [0.5, 1.0, 2.0, 3.0],
        "phase_values": [0, 45, 90, 135, 180, 225, 270, 315],
    },
    {
        "label": "A2",
        "value_name": "A2_value",
        "amp_field": "A2_amp",
        "phase_field": "A2_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 30, 60, 90, 120],
    },
    {
        "label": "A3",
        "value_name": "A3_value",
        "amp_field": "A3_amp",
        "phase_field": "A3_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
    {
        "label": "S3/C32",
        "value_name": "S3_value",
        "amp_field": "S3_amp",
        "phase_field": "S3_phase",
        "amp_values": [1, 4, 8, 16],
        "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5],
    },
]

SWEEP_SPECS = SCALAR_SWEEP_SPECS + HARMONIC_SWEEP_SPECS

COMBINATION_FIELDS = (
    "sweep_label",
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)

base_cases = []
for spec in SCALAR_SWEEP_SPECS:
    for value in spec["input_values"]:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = spec["label"]
        params[spec["input_field"]] = value
        base_cases.append(params)

for spec in HARMONIC_SWEEP_SPECS:
    for amp in spec["amp_values"]:
        for phase in spec["phase_values"]:
            params = dict(BASELINE_PARAMETERS)
            params["sweep_label"] = spec["label"]
            params[spec["amp_field"]] = amp
            params[spec["phase_field"]] = phase
            base_cases.append(params)

for c1_value in C1_C3_GRID_C1_VALUES:
    for c3_value in C1_C3_GRID_C3_VALUES:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = C1_C3_GRID_LABEL
        params["C1"] = c1_value
        params["C3"] = c3_value
        base_cases.append(params)

for a1_amp in A1_S3_GRID_A1_AMP_VALUES:
    for s3_amp in A1_S3_GRID_S3_AMP_VALUES:
        for a1_phase in A1_S3_GRID_A1_PHASE_VALUES:
            for s3_phase in A1_S3_GRID_S3_PHASE_VALUES:
                params = dict(BASELINE_PARAMETERS)
                params["sweep_label"] = A1_S3_GRID_LABEL
                params["A1_amp"] = a1_amp
                params["A1_phase"] = a1_phase
                params["S3_amp"] = s3_amp
                params["S3_phase"] = s3_phase
                base_cases.append(params)

parameters = []
for base_case in base_cases:
    for c1_offset in C1_OFFSETS:
        params = dict(base_case)
        params["C1_offset"] = c1_offset
        parameters.append(params)

print("base cases, including coupled grids:", len(base_cases))
print("simulated probe images, including C1 offsets:", len(parameters))
for spec in SWEEP_SPECS:
    count = sum(1 for case in base_cases if case["sweep_label"] == spec["label"])
    print(f"  {spec['label']}: {count} base cases")

## 6. Run Probe Simulation

In [ ]:
config = SimulationConfig(
    pix_dim=(256, 256),
    real_dim=(1280, 1280),
    app=30,
    sigma=2,
)

probe_images, selected = run_simulation(config, parameters)
# `selected` is rebuilt by the simulation backend from numeric fields only.
# Keep the original records for sweep labels and under/over pairing metadata.
simulation_records = [dict(params) for params in parameters]
probe_np = asnumpy(probe_images)
print("probe image stack:", probe_np.shape)
print("metadata records:", len(simulation_records))
print("intensity range:", float(np.min(probe_np)), float(np.max(probe_np)))
print("per-image intensity sums, min/max:", float(np.min(np.sum(probe_np, axis=(0, 1)))), float(np.max(np.sum(probe_np, axis=(0, 1)))))

NPZ_PATH = OUTPUT_DIR / "uno_relationship_probe_images.npz"
save_npz(NPZ_PATH, probe_images, selected)
print("saved:", NPZ_PATH)


## 7. Uno Formula Implementation

The value definitions are imported from `aberration_simulation.uno_conventions`:

- `compute_line_characteristics()` computes `Xigma`, `Mu`, and `Rho`.
- `compute_uno_values()` computes `Cdf_value`, `C1_value`, `A1_value`, `B2_value`, `A2_value`, `Cs_value`, `C3_value`, `S3_value`, and `A3_value`.
- `select_under_over_pairs()` pairs the `C1_offset=-909 nm` and `C1_offset=+909 nm` probes.

The formula source of truth is now the Python module, not this notebook.

## 8. Compute Uno Coefficients From Line Profiles

In [ ]:
pairs = select_under_over_pairs(
    simulation_records,
    COMBINATION_FIELDS,
    under_focus_c1_offset=UNDER_FOCUS_C1_OFFSET,
    over_focus_c1_offset=OVER_FOCUS_C1_OFFSET,
)
rows = []
print("under/over pairs:", len(pairs))

for case_index, (params, under_index, over_index) in enumerate(pairs):
    stack = probe_images[:, :, [under_index, over_index]]
    profiles, coords = extract_line_profiles_from_stack(
        stack,
        num_lines=NUM_PROFILE_LINES,
        radius=PROFILE_RADIUS_PIXELS,
    )
    angles_deg = np.asarray(coords["angles_deg"], dtype=float)

    # These functions accept CuPy arrays, so profile-characteristic and Uno-value
    # calculations stay GPU-vectorized until values are written to CSV columns.
    under_chars = compute_line_characteristics(profiles[:, :, 0], PROFILE_RADIUS_PIXELS)
    over_chars = compute_line_characteristics(profiles[:, :, 1], PROFILE_RADIUS_PIXELS)
    uno_values = compute_uno_values(under_chars, over_chars, angles_deg)

    row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
    row["case_index"] = case_index
    row["under_index"] = under_index
    row["over_index"] = over_index
    row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
    row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
    for name, value in uno_values.items():
        add_complex_columns(row, name, value)
    rows.append(row)

CSV_PATH = OUTPUT_DIR / "uno_coefficient_relationships.csv"
with CSV_PATH.open("w", newline="") as handle:
    fieldnames = list(rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("computed paired cases:", len(rows))
row_counts = {spec["label"]: sum(row["sweep_label"] == spec["label"] for row in rows) for spec in SWEEP_SPECS}
row_counts[C1_C3_GRID_LABEL] = sum(row["sweep_label"] == C1_C3_GRID_LABEL for row in rows)
row_counts[A1_S3_GRID_LABEL] = sum(row["sweep_label"] == A1_S3_GRID_LABEL for row in rows)
print("rows by sweep:", row_counts)
missing_sweeps = [label for label, count in row_counts.items() if count == 0]
if missing_sweeps:
    raise ValueError(f"No computed rows for sweeps: {missing_sweeps}")
print("saved:", CSV_PATH)

## 9. Relationship and Coupled-Response Plots

In [ ]:
plot_paths = []
plot_paths.extend(plot_relationships(rows, SCALAR_SWEEP_SPECS, HARMONIC_SWEEP_SPECS, OUTPUT_DIR))
plot_paths.append(plot_c1_c3_coupling_maps(rows, C1_C3_GRID_LABEL, OUTPUT_DIR))
plot_paths.extend(plot_a1_s3_coupling_maps(rows, A1_S3_GRID_LABEL, OUTPUT_DIR))

## 10. Pairwise Uniqueness Diagnostics

In [ ]:
def row_float(row, name):
    return float(row[name])


def complex_coefficient_vector(row, prefix, order):
    amp = row_float(row, f"{prefix}_amp")
    phase = np.deg2rad(order * row_float(row, f"{prefix}_phase"))
    return np.asarray([amp * np.cos(phase), amp * np.sin(phase)], dtype=float)


def unique_rows_by_input(selected_rows, input_vector_fn, decimals=8):
    unique_rows = []
    seen = set()
    for row in selected_rows:
        key = tuple(np.round(input_vector_fn(row), decimals))
        if key in seen:
            continue
        seen.add(key)
        unique_rows.append(row)
    return unique_rows


def pairwise_uniqueness_metrics(selected_rows, input_vector_fn, output_vector_fn):
    input_vectors = np.asarray([input_vector_fn(row) for row in selected_rows], dtype=float)
    output_vectors = np.asarray([output_vector_fn(row) for row in selected_rows], dtype=float)
    pair_input_distances = []
    pair_output_distances = []
    nearest_output_distance = np.full(len(selected_rows), np.nan, dtype=float)
    nearest_input_distance = np.full(len(selected_rows), np.nan, dtype=float)
    nearest_index = np.full(len(selected_rows), -1, dtype=int)

    for i in range(len(selected_rows)):
        best_distance = np.inf
        for j in range(len(selected_rows)):
            if i == j:
                continue
            input_distance = float(np.linalg.norm(input_vectors[i] - input_vectors[j]))
            output_distance = float(np.linalg.norm(output_vectors[i] - output_vectors[j]))
            if j > i:
                pair_input_distances.append(input_distance)
                pair_output_distances.append(output_distance)
            if output_distance < best_distance:
                best_distance = output_distance
                nearest_output_distance[i] = output_distance
                nearest_input_distance[i] = input_distance
                nearest_index[i] = j

    return {
        "input_vectors": input_vectors,
        "output_vectors": output_vectors,
        "pair_input_distances": np.asarray(pair_input_distances, dtype=float),
        "pair_output_distances": np.asarray(pair_output_distances, dtype=float),
        "nearest_output_distance": nearest_output_distance,
        "nearest_input_distance": nearest_input_distance,
        "nearest_index": nearest_index,
    }


def plot_coupling_uniqueness_diagnostics(
    label,
    input_vector_fn,
    output_vector_fn,
    nearest_map_fn,
    output_name,
):
    selected_rows = [row for row in rows if row["sweep_label"] == label]
    if not selected_rows:
        raise ValueError(f"No rows found for {label}.")
    unique_rows = unique_rows_by_input(selected_rows, input_vector_fn)
    metrics = pairwise_uniqueness_metrics(unique_rows, input_vector_fn, output_vector_fn)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), squeeze=False)
    axes[0, 0].scatter(
        metrics["pair_input_distances"],
        metrics["pair_output_distances"],
        s=12,
        alpha=0.35,
    )
    min_pair_index = int(np.nanargmin(metrics["pair_output_distances"]))
    axes[0, 0].scatter(
        metrics["pair_input_distances"][min_pair_index],
        metrics["pair_output_distances"][min_pair_index],
        color="red",
        s=42,
        label="nearest output pair",
    )
    axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[0, 0].set_title("pairwise separation")
    axes[0, 0].set_xlabel("input-space distance")
    axes[0, 0].set_ylabel("output-space distance")
    axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend(fontsize=8)

    nearest_map_fn(axes[0, 1], unique_rows, metrics)
    fig.suptitle(
        f"Uniqueness diagnostic: {label} | unique inputs={len(unique_rows)}",
        fontsize=12,
    )
    fig.tight_layout()
    plot_path = OUTPUT_DIR / output_name
    fig.savefig(plot_path, dpi=180, bbox_inches="tight")
    plt.show()

    min_output_index = int(np.nanargmin(metrics["nearest_output_distance"]))
    neighbor_index = int(metrics["nearest_index"][min_output_index])
    print("saved:", plot_path)
    print(
        label,
        "nearest output distance=", float(metrics["nearest_output_distance"][min_output_index]),
        "input distance for that pair=", float(metrics["nearest_input_distance"][min_output_index]),
    )
    print("  row:", unique_rows[min_output_index])
    print("  nearest row:", unique_rows[neighbor_index])
    return plot_path


def c1_c3_input_vector(row):
    return np.asarray([row_float(row, "C1"), row_float(row, "C3")], dtype=float)


def c1_c3_output_vector(row):
    return np.asarray([row_float(row, "C1_value_real"), row_float(row, "C3_value_real")], dtype=float)


def draw_c1_c3_nearest_map(axis, unique_rows, metrics):
    c1_values = np.asarray(sorted({row_float(row, "C1") for row in unique_rows}), dtype=float)
    c3_values = np.asarray(sorted({row_float(row, "C3") for row in unique_rows}), dtype=float)
    data = np.full((len(c3_values), len(c1_values)), np.nan, dtype=float)
    for row, distance in zip(unique_rows, metrics["nearest_output_distance"]):
        iy = int(np.where(np.isclose(c3_values, row_float(row, "C3")))[0][0])
        ix = int(np.where(np.isclose(c1_values, row_float(row, "C1")))[0][0])
        data[iy, ix] = distance
    image = axis.imshow(
        data,
        origin="lower",
        aspect="auto",
        extent=[float(c1_values.min()), float(c1_values.max()), float(c3_values.min()), float(c3_values.max())],
        cmap="magma",
    )
    axis.set_title("nearest output distance")
    axis.set_xlabel("input C1")
    axis.set_ylabel("input C3")
    axis.set_xticks(c1_values)
    axis.set_yticks(c3_values)
    axis.grid(color="white", alpha=0.25, linewidth=0.7)
    axis.figure.colorbar(image, ax=axis, label="nearest output distance")


def a1_s3_input_vector(row):
    return np.concatenate([
        complex_coefficient_vector(row, "A1", order=2),
        complex_coefficient_vector(row, "S3", order=2),
    ])


def a1_s3_output_vector(row):
    return np.asarray([
        row_float(row, "A1_value_real"),
        row_float(row, "A1_value_imag"),
        row_float(row, "S3_value_real"),
        row_float(row, "S3_value_imag"),
    ], dtype=float)


def draw_a1_s3_nearest_summary(axis, unique_rows, metrics):
    total_amp = np.asarray([
        np.hypot(row_float(row, "A1_amp"), row_float(row, "S3_amp"))
        for row in unique_rows
    ], dtype=float)
    scatter = axis.scatter(
        metrics["nearest_input_distance"],
        metrics["nearest_output_distance"],
        c=total_amp,
        cmap="viridis",
        s=46,
        alpha=0.85,
    )
    axis.axhline(0, color="black", linestyle="--", linewidth=1)
    axis.set_title("nearest output neighbor per input")
    axis.set_xlabel("input distance to nearest output neighbor")
    axis.set_ylabel("nearest output distance")
    axis.grid(alpha=0.3)
    axis.figure.colorbar(scatter, ax=axis, label="sqrt(A1_amp^2 + S3_amp^2)")


c1_c3_uniqueness_path = plot_coupling_uniqueness_diagnostics(
    C1_C3_GRID_LABEL,
    c1_c3_input_vector,
    c1_c3_output_vector,
    draw_c1_c3_nearest_map,
    "uniqueness_c1_c3.png",
)

a1_s3_uniqueness_path = plot_coupling_uniqueness_diagnostics(
    A1_S3_GRID_LABEL,
    a1_s3_input_vector,
    a1_s3_output_vector,
    draw_a1_s3_nearest_summary,
    "uniqueness_a1_s3_complex.png",
)
plot_paths.extend([c1_c3_uniqueness_path, a1_s3_uniqueness_path])


## 11. Summary Grid

In [ ]:
summary_plot_path = plot_relationship_summary(rows, SWEEP_SPECS, OUTPUT_DIR)

## 12. Probe Shape Gallery

In [ ]:
probe_shape_paths = plot_probe_shape_galleries(
    rows,
    probe_np,
    SWEEP_SPECS,
    OUTPUT_DIR,
    under_focus_c1_offset=UNDER_FOCUS_C1_OFFSET,
    over_focus_c1_offset=OVER_FOCUS_C1_OFFSET,
)

## 13. Download Bundle

In [ ]:
EXPECTED_RELATIONSHIP_OUTPUTS = [
    "relationship_a1_s3_coupling_a1_value_abs.png",
    "relationship_a1_s3_coupling_s3_value_abs.png",
    "uniqueness_c1_c3.png",
    "uniqueness_a1_s3_complex.png",
]
missing_outputs = [
    name for name in EXPECTED_RELATIONSHIP_OUTPUTS
    if not (OUTPUT_DIR / name).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Expected relationship outputs were not generated before download: "
        + ", ".join(missing_outputs)
    )
print("verified expected relationship outputs:")
for name in EXPECTED_RELATIONSHIP_OUTPUTS:
    print(" ", OUTPUT_DIR / name)


In [ ]:
REMOTE_DOWNLOAD_DIR = ROOT / "Downloads from Colab"
REMOTE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = OUTPUT_DIR / "uno_coefficient_relationships.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(CSV_PATH, CSV_PATH.name)
    zf.write(summary_plot_path, summary_plot_path.name)
    for path in plot_paths:
        zf.write(path, path.name)
    for path in probe_shape_paths:
        zf.write(path, path.name)
print("saved:", ZIP_PATH)

REMOTE_ZIP_PATH = REMOTE_DOWNLOAD_DIR / ZIP_PATH.name
REMOTE_CSV_PATH = REMOTE_DOWNLOAD_DIR / CSV_PATH.name
REMOTE_NPZ_PATH = REMOTE_DOWNLOAD_DIR / NPZ_PATH.name
for source, target in [
    (ZIP_PATH, REMOTE_ZIP_PATH),
    (CSV_PATH, REMOTE_CSV_PATH),
    (NPZ_PATH, REMOTE_NPZ_PATH),
]:
    target.write_bytes(source.read_bytes())
    print("copied inside Colab VM:", target)

print()
print("Important: the copied folder above is inside the remote Colab runtime, not your local Mac repo folder.")
print("Colab cannot directly write to /Users/.../Downloads from Colab on the local machine.")
print("The browser or VS Code controls where files.download() lands locally.")

try:
    from google.colab import files
    files.download(str(REMOTE_ZIP_PATH))
    files.download(str(REMOTE_CSV_PATH))
except Exception as exc:
    print("Browser download skipped or unsupported in this frontend:", exc)
    print("Download manually from the Colab file browser:", REMOTE_DOWNLOAD_DIR)
